# AwareLiquid M2 — MT-LNN v2.0 Pretraining

**125M-class MT-LNN from-scratch** with all v2.0 bio-inspired modules:
- Competitive GWT-B (global workspace)
- Predictive coding / world model
- Hebbian plasticity

**Runtime:** ~7h on T4 for 5000 steps (~41M tokens on WikiText-103)  
**Output:** `checkpoints/serve.pt` — drop-in for `serve/server.py`

### Before running:
1. `Runtime > Change runtime type > T4 GPU`
2. Optionally mount Google Drive (Cell 2) to persist checkpoints across sessions

In [ ]:
# ── Cell 1: GPU check ──────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    dev = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {dev}  |  sm_{cap[0]}{cap[1]}  |  {mem:.1f} GB VRAM')
else:
    print('NO GPU -- switch runtime to T4!')
    raise SystemExit('No GPU')

In [ ]:
# ── Cell 2: (optional) mount Google Drive to persist checkpoints ───────────
USE_DRIVE = True   # set False to skip Drive mount

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/awareliquid_m2/checkpoints'
else:
    CKPT_DIR = '/content/checkpoints'

import os
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoints -> {CKPT_DIR}')

In [ ]:
# ── Cell 3: clone repo & install deps ─────────────────────────────────────
import subprocess, sys, os

REPO = 'https://github.com/everest-an/M1.git'
DIR  = '/content/M1'
if not os.path.exists(DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, DIR], check=True)
os.chdir(DIR)
sys.path.insert(0, DIR)
print(f'Repo at {DIR}')

# Pascal (sm_60) compat -- same fix as the Kaggle kernel
cap = torch.cuda.get_device_capability(0)
if cap[0] < 7:
    print(f'Pascal GPU detected -- installing torch 2.6.0+cu124')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch==2.6.0', '--index-url',
                    'https://download.pytorch.org/whl/cu124'], check=True)

# Pin torch, install extras
torch_ver = subprocess.check_output(
    [sys.executable, '-c',
     "import torch;print(torch.__version__.split('+')[0])"]
).decode().strip()
constraints = '/tmp/pip-constraints.txt'
with open(constraints, 'w') as f:
    f.write(f'torch=={torch_ver}\n')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-c', constraints,
                'datasets', 'transformers', 'tokenizers',
                'tqdm', 'einops', 'wandb'], check=True)
print(f'torch {torch_ver} ready')

In [ ]:
# ── Cell 4: tokenise WikiText-103 (once, ~7 min) ──────────────────────────
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(os.path.join(DATA_DIR, 'meta.json')):
    print('Tokenising WikiText-103 (gpt2 tokenizer)...')
    subprocess.run(
        [sys.executable, 'prepare_data.py',
         '--dataset', 'wikitext', '--config', 'wikitext-103-raw-v1',
         '--tokenizer', 'gpt2', '--out_dir', DATA_DIR],
        check=True
    )
else:
    print('Reusing existing tokenised data/')

import json
meta = json.load(open(os.path.join(DATA_DIR, 'meta.json')))
print(f"Train tokens: {meta.get('train_tokens', '?'):,}")

In [ ]:
# ── Cell 5: resume detection ───────────────────────────────────────────────
resume_args = []
candidates = [
    os.path.join(CKPT_DIR, 'last.pt'),
    os.path.join(CKPT_DIR, 'final.pt'),
]
existing = [c for c in candidates if os.path.exists(c)]
if existing:
    ckpt = max(existing, key=os.path.getmtime)
    resume_args = ['--resume', ckpt]
    ck = torch.load(ckpt, map_location='cpu', weights_only=False)
    print(f'Resuming from {ckpt}  (step {ck.get("step", "?")}, '
          f'loss {ck.get("loss", "?")})')
else:
    print('Fresh start (no checkpoint found)')

In [ ]:
# ── Cell 6: TRAIN ─────────────────────────────────────────────────────────
# 5000 steps x global-batch 8192 tok = ~41M tokens, ~7h on T4
# Raise M2_STEPS to 10000 for a second session (attach Drive checkpoint).
STEPS      = int(os.environ.get('M2_STEPS',      '5000'))
BATCH      = int(os.environ.get('M2_BATCH',      '4'))
GRAD_ACCUM = int(os.environ.get('M2_GRAD_ACCUM', '4'))   # global 16, 8192 tok/step
METRICS    = '/content/metrics.jsonl'

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

cmd = [
    sys.executable, 'train.py',
    '--data_dir',       DATA_DIR,
    '--ckpt_dir',       CKPT_DIR,
    '--metrics_jsonl',  METRICS,
    '--metrics_every',  '50',
    '--wandb_run_name', 'awareliquid-m2-colab',
    # 125M architecture (Tensor-Core-aligned: 832 = 13 x 64)
    '--d_model',    '832',
    '--n_layers',   '12',
    '--n_heads',    '13',
    '--n_kv_heads', '1',
    '--seq_len',    '512',
    # schedule
    '--batch',          str(BATCH),
    '--grad_accum',     str(GRAD_ACCUM),
    '--lr',             '6e-4',
    '--warmup_steps',   '120',
    '--steps',          str(STEPS),
    '--log_every',      '50',
    '--eval_every',     '400',
    '--eval_batches',   '40',
    '--save_every',     '400',
    # v2.0 modules
    '--competitive_gwtb', '--n_bids', '3',
    '--world_model', '--world_model_weight', '0.01',
    '--world_model_grad_clip', '1.0',
    '--hebbian', '--hebbian_lr', '1e-4',
] + resume_args

print('CMD:', ' '.join(cmd[:10]), '...')
print(f'Steps: {STEPS}  |  global batch: {BATCH*GRAD_ACCUM}  |  '
      f'{BATCH*GRAD_ACCUM*512:,} tok/step')
subprocess.run(cmd, check=True)

In [ ]:
# ── Cell 7: post-run -- write serve.pt + summary ───────────────────────────
import shutil, time

final = os.path.join(CKPT_DIR, 'final.pt')
last  = os.path.join(CKPT_DIR, 'last.pt')
serve = os.path.join(CKPT_DIR, 'serve.pt')

if os.path.exists(final):
    shutil.copyfile(final, last)
    print(f'Copied final.pt -> last.pt')

    ck = torch.load(final, map_location='cpu', weights_only=False)
    slim = {'config': ck['config'], 'model_state': ck['model_state'],
            'step': ck.get('step'), 'loss': ck.get('loss')}
    torch.save(slim, serve)
    sz = os.path.getsize(serve) / 1e6
    print(f'serve.pt: {sz:.0f} MB  |  step {slim["step"]}  |  loss {slim["loss"]:.4f}')

    # Download serve.pt directly from Colab
    if not USE_DRIVE:
        from google.colab import files
        files.download(serve)
        print('serve.pt download triggered')
    else:
        print(f'serve.pt saved to Drive: {serve}')
else:
    print('final.pt not found -- training may have stopped early')
    ckpts = sorted([f for f in os.listdir(CKPT_DIR) if f.endswith('.pt')])
    print('Available checkpoints:', ckpts)

In [ ]:
# ── Cell 8: (optional) quick metrics plot ─────────────────────────────────
import json
import matplotlib.pyplot as plt

rows = []
if os.path.exists(METRICS):
    with open(METRICS) as f:
        for line in f:
            try: rows.append(json.loads(line))
            except: pass

if rows:
    steps  = [r['step']     for r in rows if 'step'     in r]
    losses = [r['loss']     for r in rows if 'loss'     in r]
    val_pp = [r['val_ppl']  for r in rows if 'val_ppl'  in r]
    val_st = [r['step']     for r in rows if 'val_ppl'  in r]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(steps[:len(losses)], losses, lw=0.8)
    ax1.set(title='Train loss', xlabel='step', ylabel='loss')
    ax2.plot(val_st, val_pp, 'o-', lw=1.2, ms=4, color='orange')
    ax2.set(title='Val perplexity', xlabel='step', ylabel='PPL')
    plt.tight_layout()
    plt.savefig('/content/metrics_plot.png', dpi=120)
    plt.show()
    print(f'{len(rows)} metric rows | final loss {losses[-1]:.4f} | '
          f'val PPL {val_pp[-1]:.1f}' if val_pp else '')
else:
    print('No metrics yet')